In [1]:
# Importing required libraries
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torchvision
from torch.utils.data import DataLoader

In [2]:
# Importing images (from a folder containing both good and faulty images)
path = '/Users/dcac/Data/computer_vision/images_anomaly_detection/bottle_mixed/'
# path = '/Users/dcac/Data/computer_vision/images_anomaly_detection/metal_nut_mixed/'

In [3]:
# Transformations (e.g., cropping and normalization) required by torchvision pretrained models
transformations = transforms.Compose([transforms.RandomResizedCrop(224),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

In [4]:
# Creating a data loader for torch model
dataset = datasets.ImageFolder(path, transform=transformations)
train_loader = DataLoader(dataset, batch_size=len(dataset), shuffle=True)
images, labels = next(iter(train_loader))

In [5]:
# Feature extractor: removing last layer from pretrained resnet
model = torchvision.models.resnet18(pretrained=True)
new_model = torch.nn.Sequential(*(list(model.children())[:-1]))
print(new_model)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [6]:
# Getting the extracted features
output = new_model(images)
output = output.reshape(-1, 512)

In [7]:
# Further reducing the dimensionality from 512 to 100 (retaining 90% of the total explained variance)
preprocessed_features = pd.DataFrame(output.cpu().detach().numpy())
pca = PCA(n_components=100, random_state=42)  # 100 PCs = 90% of explained variance
pca.fit(preprocessed_features)
preprocessed_features = pd.DataFrame(pca.transform(preprocessed_features))

In [8]:
# Exporting labeled csv to train a classification model
multiclass_y = labels.cpu().detach().numpy()
preprocessed_features["y"] = multiclass_y
binary_y = np.where(multiclass_y > 0, 1, multiclass_y)
preprocessed_features["y"] = binary_y
preprocessed_features.to_csv("preprocessed_features_bottles.csv", index=False)